In [ ]:
!pip install torch gpytorch botorch

## Define Objective Function

In [12]:
import torch
import numpy as np
import gpytorch

device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

In [13]:
# goal: get specific RGB values (ie. UW purple)
# input: CMY volumes, output: RGB values

# Define the target RGB values and the max volume constraint
target_rgb = torch.tensor([
    [75,46,131]
], dtype=torch.float64)

def feasibility_constraint(cmyk_volumes):
    """Feasibility constraint: sum of CMYK volumes should not exceed max_volume."""
    return cmyk_volumes.sum(axis=1) - max_volume

# Objective function with a hard feasibility constraint
def objective_from_cmyk(batch_size=10):
    num_targets = target_rgb.size(0)  # Number of target RGB values

    # Latin Hypercube Sampling
    def lhs(n, samples):
        result = np.zeros((samples, n))
        for i in range(n):
            perm = np.random.permutation(samples) + 1
            result[:, i] = (perm - np.random.rand(samples)) / samples
        return result

    # Generate CMYK volumes and scale to max volume
    cmyk_volumes = lhs(4, batch_size) * max_volume
    V_c, V_m, V_y, V_k = cmyk_volumes[:, 0], cmyk_volumes[:, 1], cmyk_volumes[:, 2], cmyk_volumes[:, 3]

    # Convert CMYK volumes to RGB values using the CMYK to RGB formula
    R = 255 * (1 - V_c / max_volume) * (1 - V_k / max_volume)
    G = 255 * (1 - V_m / max_volume) * (1 - V_k / max_volume)
    B = 255 * (1 - V_y / max_volume) * (1 - V_k / max_volume)
    generated_rgb = torch.tensor(np.column_stack([R, G, B]), dtype=torch.float64)

    # Apply hard feasibility constraint: sum of CMYK volumes <= max_volume
    lhs_constraint = torch.tensor(cmyk_volumes.sum(axis=1) <= max_volume, dtype=torch.float64)
    feasibility_mask = lhs_constraint  # Hard mask: 1 for feasible, 0 for infeasible

    # Calculate error for each target RGB in the batch
    errors = []
    for target_rgb in target_rgb:
        error = torch.sqrt(torch.sum((generated_rgb - target_rgb) ** 2, dim=1))
        errors.append(error)

    # Stack errors for all targets, then apply hard feasibility mask
    errors = torch.stack(errors, dim=1)  # shape: (batch_size, num_targets)
    weighted_errors = errors * feasibility_mask.unsqueeze(1)  # Apply hard feasibility mask
    mean_error = weighted_errors.mean()  # Overall mean error for feasible samples

    return mean_error, generated_rgb, cmyk_volumes, errors, feasibility_mask

## Set up Gaussian Process Model

### Generate Training Data

In [19]:
import torch
from typing import Optional
from botorch.acquisition import ExpectedImprovement, ConstrainedMCObjective
from botorch.acquisition.objective import GenericMCObjective
from botorch.models import SingleTaskGP, ModelListGP
from botorch.models.transforms.input import Normalize
from botorch.sampling import MCSampler  # Generic sampler for MC sampling
from gpytorch.mlls import SumMarginalLogLikelihood
from botorch.test_functions import Hartmann

# Define neg_hartmann6 as the Hartmann6 function, with negation to turn it into a maximization objective
neg_hartmann6 = Hartmann(negate=True)

NOISE_SE = 0.25
train_yvar = torch.tensor(NOISE_SE**2)

# Define the objective and constraint callables
def obj_callable(Z: torch.Tensor, X: Optional[torch.Tensor] = None):
    return Z[..., 0]

def constraint_callable(Z):
    return Z[..., 1]  # Ensure this constraint returns negative values for feasibility

objective = GenericMCObjective(objective=obj_callable)

# Generate initial data
def generate_initial_data(n=10):
    train_x = torch.rand(10, 6)  # Sampled within [0, 1] range
    exact_obj = neg_hartmann6(train_x).unsqueeze(-1)
    exact_con = feasibility_constraint(train_x).unsqueeze(-1)
    train_obj = exact_obj + NOISE_SE * torch.randn_like(exact_obj)
    train_con = exact_con + NOISE_SE * torch.randn_like(exact_con)
    best_observed_value = weighted_obj(train_x).max().item()
    return train_x, train_obj, train_con, best_observed_value

# Initialize model
def initialize_model(train_x, train_obj, train_con, state_dict=None):
    model_obj = SingleTaskGP(
        train_x,
        train_obj,
        train_yvar.expand_as(train_obj),
        input_transform=Normalize(d=train_x.shape[-1]),
    )
    model_con = SingleTaskGP(
        train_x,
        train_con,
        train_yvar.expand_as(train_con),
        input_transform=Normalize(d=train_x.shape[-1]),
    )
    model = ModelListGP(model_obj, model_con)
    mll = SumMarginalLogLikelihood(model.likelihood, model)
    if state_dict is not None:
        model.load_state_dict(state_dict)
    return mll, model

# Define constrained objective
constrained_objective = ConstrainedMCObjective(
    objective=obj_callable,
    constraints=[constraint_callable],  # This constraint should evaluate to <= 0 for feasible points
)

# Acquisition function with Expected Improvement
def initialize_acquisition_function(model):
    sampler = MCSampler(num_samples=500)  # Use generic MCSampler as a replacement
    ei = ExpectedImprovement(model=model.models[0], best_f=0.0, sampler=sampler)  # EI on objective model
    return ei

# Main optimization loop (simplified for demonstration)
train_x, train_obj, train_con, best_observed_value = generate_initial_data()
mll, model = initialize_model(train_x, train_obj, train_con)
ei = initialize_acquisition_function(model)

# Assume we perform one optimization step to acquire a new point
new_x = ei.forward(train_x).argmax().unsqueeze(0)  # EI suggests next point

NameError: name 'weighted_obj' is not defined

## Optimize using BO

In [ ]:
# define helper function



## Visualizing convergence of BO

In [ ]:
!pip install matplotlib

In [ ]:
"""
Exploitation:
Visualize how RGB values converge over iterations of the BO process. Basically, it tracks
the best RGB output obtained after each iteration of optimization.
"""

def plot_convergence(iterations, best_rgb_values):
    plt.figure(figsize=(12, 6))

    plt.subplot(3, 1, 1)
    plt.plot(iterations, best_rgb_values[:, 0], label='Red', color='red')
    plt.ylabel('Red Channel Value')
    plt.title('Convergence of RGB Values')
    plt.grid()
    plt.legend()

    plt.subplot(3, 1, 2)
    plt.plot(iterations, best_rgb_values[:, 1], label='Green', color='green')
    plt.ylabel('Green Channel Value')
    plt.grid()
    plt.legend()

    plt.subplot(3, 1, 3)
    plt.plot(iterations, best_rgb_values[:, 2], label='Blue', color='blue')
    plt.ylabel('Blue Channel Value')
    plt.xlabel('Iterations')
    plt.grid()
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_convergence(iterations, best_rgb_values)

In [ ]:
"""
Exploration:
Visualizes the predicted RGB values based on CMY values as predicted by the Gaussian
Process model. Basically, it helps in understanding the model's output across the 
entire input space.
"""

# Define a grid of CMY values for visualization
num_points = 100
cmy_grid = torch.linspace(0.0, 1.0, num_points)
cmy_mesh = torch.cartesian_prod(cmy_grid, cmy_grid, cmy_grid)

# Predict RGB values using the GP model
with torch.no_grad():
    gp.eval()
    pred_mean = gp.posterior(cmy_mesh).mean.detach()
    pred_var = gp.posterior(cmy_mesh).variance.detach()

# Visualize predictions
def plot_gp_predictions(cmy_mesh, pred_mean, pred_var):
    # Convert CMY to RGB for a scatter plot
    rgb_values = pred_mean.numpy()
    
    plt.figure(figsize=(10, 10))
    scatter = plt.scatter(rgb_values[:, 0], rgb_values[:, 1], c=rgb_values, s=10)
    plt.colorbar(scatter, label='RGB Value')
    plt.xlabel('Red Channel Value')
    plt.ylabel('Green Channel Value')
    plt.title('GP Predictions in RGB Space')
    plt.grid()
    plt.show()

plot_gp_predictions(cmy_mesh, pred_mean, pred_var)